In [ ]:
#???????????
import numpy as np
import lightgbm as lgb
import pandas as pd
from kaggle.competitions import twosigmanews
import matplotlib.pyplot as plt
import random
from datetime import datetime, date
from xgboost import XGBClassifier
from sklearn import model_selection
from sklearn.metrics import mean_squared_error
import time

In [ ]:
#?????????
from kaggle.competitions import twosigmanews
env = twosigmanews.make_env()
print('Done!')

In [ ]:
#???????????????????????
(market_train_df, news_train_df) = env.get_training_data()

In [ ]:
market_train_df.head()

In [ ]:
#time????????
market_train_df['time'] = market_train_df['time'].dt.date
#2010?~????????
market_train_df = market_train_df.loc[market_train_df['time']>=date(2010, 1, 1)]

In [ ]:
market_train_df.head()

In [ ]:
#????????????????
from multiprocessing import Pool

#create_lag(?????????????????????)
#??????????????????????
def create_lag(df_code,n_lag=[3,7,14,],shift_size=1):

    #df_code????????????code????
    code = df_code['assetCode'].unique()
    
    #return_features(????)??col???
    for col in return_features:
        #n_lag?window????????
        for window in n_lag:
            #window????????????????????????????????
            rolled = df_code[col].shift(shift_size).rolling(window=window)
            #???????
            lag_mean = rolled.mean()
            #???
            lag_max = rolled.max()
            #???
            lag_min = rolled.min()
            #????
            lag_std = rolled.std()
            #col??_lag_window??_mean??????df_code??????
            df_code['%s_lag_%s_mean'%(col,window)] = lag_mean
            df_code['%s_lag_%s_max'%(col,window)] = lag_max
            df_code['%s_lag_%s_min'%(col,window)] = lag_min
            #??????????
#             df_code['%s_lag_%s_std'%(col,window)] = lag_std
    #df_code????????-1?????
    return df_code.fillna(-1)


#generate_lag_features(?????????)
def generate_lag_features(df,n_lag = [3,7,14]):
    #features(????)????????????(????????????????)
    features = ['time', 'assetCode', 'assetName', 'volume', 'close', 'open',
       'returnsClosePrevRaw1', 'returnsOpenPrevRaw1',
       'returnsClosePrevMktres1', 'returnsOpenPrevMktres1',
       'returnsClosePrevRaw10', 'returnsOpenPrevRaw10',
       'returnsClosePrevMktres10', 'returnsOpenPrevMktres10',
       'returnsOpenNextMktres10', 'universe']
    #assetCode(?????)?????
    assetCodes = df['assetCode'].unique()
    #??
    print(assetCodes)
    #all_df??????????
    all_df = []
    #df_codes?assetCode???(SQL??????)???
    df_codes = df.groupby('assetCode')
    #df_codes(?????)??df_code(?????)??????????????????????????
    #??
    df_codes = [df_code[1][['time','assetCode']+return_features] for df_code in df_codes]
    #df_codes??????
    print('total %s df'%len(df_codes))
    
    #cpu4????????
    pool = Pool(4)
    #create_lag?????df_codes?????????????all_df???????
    all_df = pool.map(create_lag, df_codes)
    
    #new_df?all_df???????
    new_df = pd.concat(all_df)
    #return_features????????
    new_df.drop(return_features,axis=1,inplace=True)
    #??????
    pool.close()
    #new_df???
    return new_df

In [ ]:
#?????????????????????????

# return_features(????)???????
# return_features = ['close']
#?new_df??????????????????????????????????????????5????
# new_df = generate_lag_features(market_train_df,n_lag = 5)
# ????????new_df?time assetCode?????
# market_train_df = pd.merge(market_train_df,new_df,how='left',on=['time','assetCode'])

In [ ]:
#return_features(????)???(??????????)
return_features = ['returnsClosePrevMktres10','returnsClosePrevRaw10','open','close']
#?????????
n_lag = [3,7,14]
#?????????????????????3,7,14???????????
new_df = generate_lag_features(market_train_df,n_lag=n_lag)

In [ ]:
new_df.head()

In [ ]:
#?????????????????time,assetCode????????
market_train_df = pd.merge(market_train_df,new_df,how='left',on=['time','assetCode'])

In [ ]:
#?????????????????????
print(market_train_df.columns)

In [ ]:
market_train_df.head()

In [ ]:
#mis_impute????(????)
def mis_impute(data):
    #???????????????
    for i in data.columns:
        #??????????other?????
        if data[i].dtype == "object":
            data[i] = data[i].fillna("other")
        #?????????????????
        elif (data[i].dtype == "int64" or data[i].dtype == "float64"):
            data[i] = data[i].fillna(data[i].mean())
        else:
            pass
    return data
#??????????????????
market_train_df = mis_impute(market_train_df)

In [ ]:
#data_prep(?????????????????????)
def data_prep(market_train):
    #???????????????????????????????????
    lbl = {k: v for v, k in enumerate(market_train['assetCode'].unique())}
    #???????????(??????assetCodeT??????????????)
    market_train['assetCodeT'] = market_train['assetCode'].map(lbl)
    #???????
    market_train = market_train.dropna(axis=0)
    return market_train
#??????????
market_train_df = data_prep(market_train_df)
# # shape????????????????????
print(market_train_df.shape)

In [ ]:
#assetCodeT????????????????
market_train_df.tail()

In [ ]:
#??????????????????????
from sklearn.preprocessing import LabelEncoder
#10???????????????????up???????
up = market_train_df['returnsOpenNextMktres10'] >= 0

#universe(???1???????????????)???universe???????
#???????
universe = market_train_df['universe'].values


In [ ]:
universe

In [ ]:
#d?????????
d = market_train_df['time']
#??????????????????????
#?????????????????????
#?????????????????
fcol = [c for c in market_train_df if c not in ['assetCode', 'assetCodes', 'assetCodesLen', 'assetName', 'audiences', 
                                             'firstCreated', 'headline', 'headlineTag', 'marketCommentary', 'time_x','provider', 
                                             'returnsOpenNextMktres10', 'sourceId', 'subjects', 'time', 'universe','sourceTimestamp']]
fcol

In [ ]:
#?????????????????????
X = market_train_df[fcol].values
X

In [ ]:
#returnsOpenNextMktres10?TrueFalse???????
up = up.values
up

In [ ]:
#r?TrueFalse?????????????????
r = market_train_df.returnsOpenNextMktres10.values
r

In [ ]:
# X?????(??????????????????????)
#?????????????
mins = np.min(X, axis=0)
mins

In [ ]:
#????????????
maxs = np.max(X, axis=0)
maxs

In [ ]:
#????????????
rng = maxs - mins
rng

In [ ]:
#????????????????????X???
#???????0.5 ??10%????0.9
X = 1 - ((maxs - X) / rng)
X

In [ ]:
# ????????? ???
assert X.shape[0] == up.shape[0] == r.shape[0]
print(X.shape[0])
print(up.shape[0])
print(r.shape[0])
#?????????????????????

In [ ]:
#????Xgboost????
from xgboost import XGBClassifier
from sklearn import model_selection
from sklearn.metrics import mean_squared_error
import time

In [ ]:
# X_train, X_test, up_train, up_test, r_train, r_test,u_train,u_test,d_train,d_test = model_selection.train_test_split(X, up, r,universe,d, test_size=0.25, random_state=99)
#??????????????????
#2015????????True????False???
#te = market_train_df['time']>date(2015, 1, 1)
#te
#?????????????version??????
#?????????train_test_split?????????75%??????25%????????????
# X = ???? 
# up = returnsOpenNextMktres10???TrueFalse 
#?r = returnsOpenNextMktres10??
# u = universe???1?????????
# d = ??
X_train, X_test, up_train, up_test, r_train, r_test,u_train,u_test,d_train,d_test = \
model_selection.train_test_split(X, up, r,universe,d, test_size=0.25, random_state=99)

In [ ]:
#2015????????2946738???2054539????????????
#tt = 0
#for tt,i in enumerate(te.values):
#    if i:
#        idx = tt
#        print(i,tt)
#        break
#print(idx)

In [ ]:
#?????????????????????(??????)
#????
#X_train, X_test = X[:idx],X[idx:]
#returnsOpenNextMktres10???TrueFalse
#up_train, up_test = up[:idx],up[idx:]
#returnsOpenNextMktres10??
#r_train, r_test = r[:idx],r[idx:]
#1?????????
#u_train,u_test = universe[:idx],universe[idx:]
#??
#d_train,d_test = d[:idx],d[idx:]

#LightGBM?????????
train_data = lgb.Dataset(X, label=up.astype(int))
#LightGBM??????????
test_data = lgb.Dataset(X_test, label=up_test.astype(int))

In [ ]:
# LightBGM?????????param_1 param_2??????
x_1 = [0.19000424246380565, 2452, 212, 328, 202]
x_2 = [0.19016805202090095, 2583, 213, 312, 220]
print(up_train)

####################??????? ????????????????????

def exp_loss(p,y):
    #y?????????????
    y = y.get_label()
#     p = p.get_label()
    #exp??????????????
    grad = -y*(1.0-1.0/(1.0+np.exp(-y*p)))
    hess = -(np.exp(y*p)*(y*p-1)-1)/((np.exp(y*p)+1)**2)
    
    return grad,hess
###########################################
#????????????????????
params_1 = {
        #?????????????????????
        #????
        'task': 'train',
        #????
        'boosting_type': 'gbdt',
        #regression(??)??????????binary
        'objective': 'binary',
#         'objective': 'regression',
        #?????(??????????)
        'learning_rate': x_1[0],
        #?????(?????????????????????????)
        'num_leaves': x_1[1],
        #???????
        'min_data_in_leaf': x_1[2],
#         'num_iteration': x_1[3],
        #????(??????????)
        'num_iteration': 239,
        #???????????????????????????????
        'max_bin': x_1[4],
        #?????????(0???????1??????2?????)
        'verbose': 1
    }
#2???????????????
params_2 = {
        'task': 'train',
        'boosting_type': 'gbdt',
        'objective': 'binary',
#         'objective': 'regression',
        'learning_rate': x_2[0],
        'num_leaves': x_2[1],
        'min_data_in_leaf': x_2[2],
#         'num_iteration': x_2[3],
        'num_iteration': 172,
        'max_bin': x_2[4],
        'verbose': 1
    }

gbm_1 = lgb.train(params_1,
        train_data,
#?????????????????????num_boost_round??????200????????????error rate?????????????????????
#??num_boost_round??????????????????????????????????
#???????????????133?????????
        #?????????(???????)
        num_boost_round=133,
        #?????????????
        valid_sets=test_data,
        #??????
        early_stopping_rounds=5,
#         fobj=exp_loss,
        )
#??????
gbm_2 = lgb.train(params_2,
        train_data,
        num_boost_round=133,
        valid_sets=test_data,
        early_stopping_rounds=5,
#         fobj=exp_loss,
        )

In [ ]:
#1???????2?????????????
confidence_test = (gbm_1.predict(X_test) + gbm_2.predict(X_test))/2
confidence_test

In [ ]:
#????%??????
confidence_test = (confidence_test-confidence_test.min())/(confidence_test.max()-confidence_test.min())
confidence_test

In [ ]:
#%?????-1????
confidence_test = confidence_test*2-1
print(max(confidence_test),min(confidence_test))
confidence_test

In [ ]:
# ?????????????????????????
r_test = r_test.clip(-1,1) # -1~1????????????????????????
#????????????????????????????
x_t_i = confidence_test * r_test * u_test
#???????????????????
data = {'day' : d_test, 'x_t_i' : x_t_i}
df = pd.DataFrame(data)
#?????????(??)????????1???????
x_t = df.groupby('day').sum().values.flatten()
#???????
mean = np.mean(x_t)
#????????
std = np.std(x_t)
#???????
score_test = mean / std
print(score_test)

In [ ]:
import gc
del X_train,X_test
gc.collect()

In [ ]:
#prediction
days = env.get_prediction_days()
n_days = 0
prep_time = 0
prediction_time = 0
packaging_time = 0
total_market_obs_df = []
for (market_obs_df, news_obs_df, predictions_template_df) in days:
    n_days +=1
    if (n_days%50==0):
        print(n_days,end=' ')
    t = time.time()
    market_obs_df['time'] = market_obs_df['time'].dt.date
    
    return_features = ['returnsClosePrevMktres10','returnsClosePrevRaw10','open','close']
    total_market_obs_df.append(market_obs_df)
    if len(total_market_obs_df)==1:
        history_df = total_market_obs_df[0]
    else:
        history_df = pd.concat(total_market_obs_df[-(np.max(n_lag)+1):])
    print(history_df)
    
    new_df = generate_lag_features(history_df,n_lag=[3,7,14])
    market_obs_df = pd.merge(market_obs_df,new_df,how='left',on=['time','assetCode'])
    
#     return_features = ['open']
#     new_df = generate_lag_features(market_obs_df,n_lag=[3,7,14])
#     market_obs_df = pd.merge(market_obs_df,new_df,how='left',on=['time','assetCode'])
    
    market_obs_df = mis_impute(market_obs_df)
    
    market_obs_df = data_prep(market_obs_df)
    
#     market_obs_df = market_obs_df[market_obs_df.assetCode.isin(predictions_template_df.assetCode)]
    
    X_live = market_obs_df[fcol].values
    X_live = 1 - ((maxs - X_live) / rng)
    prep_time += time.time() - t
    
    t = time.time()
    lp = (gbm_1.predict(X_live) + gbm_2.predict(X_live))/2
    prediction_time += time.time() -t
    
    t = time.time()
    
    confidence = lp
    confidence = (confidence-confidence.min())/(confidence.max()-confidence.min())
    confidence = confidence * 2 - 1
    
    preds = pd.DataFrame({'assetCode':market_obs_df['assetCode'],'confidence':confidence})
    predictions_template_df = predictions_template_df.merge(preds,how='left').drop('confidenceValue',axis=1).fillna(0).rename(columns={'confidence':'confidenceValue'})
    env.predict(predictions_template_df)
    packaging_time += time.time() - t
    
env.write_submission_file()
sub  = pd.read_csv("submission.csv")